# QDK Interop with OpenQASM

The QDK provides interoperability with OpenQASM 3 programs built upon the core QDK compiler infrastructure.

This core enables integration and local resource estimation without relying on external tools. Users are able to estimate resources for their OpenQASM programs locally (see the [resource estimation with OpenQASM sample notebook](../estimation/estimation-openqasm.ipynb)), leveraging the QDK compiler's capabilities for analysis, transformation, code generation, and simulation. This also enables the generation of QIR from OpenQASM progams leveraging the [QDKs advanced code generation capabilities](https://devblogs.microsoft.com/qsharp/integrated-hybrid-support-in-the-azure-quantum-development-kit/).

This includes support for classical instructions available in OpenQASM such as for loops, if statements, switch statements, while loops, binary expresssions, and more.

### Simulating OpenQASM programs

In [ ]:
from qdk.openqasm import run

source = """
    include "stdgates.inc";
    bit[2] c;
    qubit[2] q;
    h q[0];
    cx q[0], q[1];
    c = measure q;
"""

# We'll pass as_bitstring=True to convert bit[n] to a bitstring in the output.
# Otherwise, the output would be a list of Result values.
run(source, as_bitstring=True)

The OpenQASM programs can also be run with noise just as with Q#.

In [ ]:
from qdk import qsharp
from qdk.qsharp import DepolarizingNoise
from qdk.openqasm import run
from qdk.widgets import Histogram

source = """
    include "stdgates.inc";
    bit[2] c;
    qubit[2] q;
    h q[0];
    cx q[0], q[1];
    c = measure q;
"""

Histogram(run(source, noise=DepolarizingNoise(0.01), as_bitstring=True))


### Compiling OpenQASM to Quantum Intermediate Representation (QIR)

We can directly compile OpenQASM to QIR with the `compile` function.

In [ ]:
from qdk.openqasm import compile

source = """
    include "stdgates.inc";
    bit[2] c;
    qubit[2] q;
    h q[0];
    cx q[0], q[1];
    c = measure q;
"""

compilation = compile(source)

print(compilation)

> For parameterized circuits the `import_openqasm` function must be used to first create a Python callable. A sample parameterized circuit can be found later in this notebook.

### Run OpenQASM 3 Code in interactive session

Import the `qsharp` module from qdk.

This initializes a QDK interpreter singleton.

In [ ]:
from qdk import qsharp
qsharp.init(target_profile=qsharp.TargetProfile.Base)

With the runtime initialized, we can import an OpenQASM program as a Python callable. Here we'll compile the OpenQASM program to a callable name `"bell"`.

In [2]:
from qsharp.openqasm import import_openqasm, ProgramType

source = """
    include "stdgates.inc";
    bit[2] c;
    qubit[2] q;
    h q[0];
    cx q[0], q[1];
    c = measure q;
"""

import_openqasm(source, name="bell", program_type=ProgramType.File)

We can now import it via the QDK's Python bindings and run it:

In [ ]:
from qdk.code.qasm_import import bell
bell()

ModuleNotFoundError: No module named 'qdk.code'

Additionally, since it is defined in the runtime, we can run it directly from a Q# cell:

In [ ]:
%%qsharp
qasm_import.bell()

This also unlocks all of the other `qsharp` module functionality. Like noisy simulation. Here we'll use the `run` function showing how we can call into the program from Python and display a histogram:

In [ ]:
from qdk.qsharp import DepolarizingNoise
from qdk.openqasm import run
from qdk.widgets import Histogram

Histogram(run(bell, shots=1000, noise=DepolarizingNoise(0.01)))

We can draw the progam as a textual circuit rendering passing the Python callable into the circuit function:

In [ ]:
qsharp.circuit(bell)

In notebooks, we can do a bit better leveraging the circuit widget:

In [ ]:
from qdk.widgets import Circuit

Circuit(qsharp.circuit(bell))

And finally when getting ready to submit to hardware, we can compile the program to QIR:

In [ ]:
print(qsharp.compile(bell))

Alternatively, an OpenQASM program can be imported as an operation in Q#, where the qubits declared in OpenQASM become arguments passed into the operation. This allows the operation to be called as part of a larger Q# program that may perform additional operations before or after the imported operation.

In [ ]:
source = """
    include "stdgates.inc";
    bit[2] c;
    qubit[2] q;
    h q[0];
    cx q[0], q[1];
    c = measure q;
"""

import_openqasm(source, name="BellOp", program_type=ProgramType.Operation)

Here, we call the resulting operation from within a Q# cell by first allocating and preparing the qubits before passing them as arguments, resetting them before they are released at the end of the scope.

In [ ]:
%%qsharp
{
    use qs = Qubit[2];
    // Change the state of the second qubit before invoking the operation to show
    // an anti-correlation in the results.
    X(qs[1]);
    let result = BellOp(qs);
    ResetAll(qs);
    result
}

We can also define input for the compiled OpenQASM code so that we can parameterize input with imported callables:

In [ ]:
from qdk.qsharp import init, TargetProfile

source = """
include "stdgates.inc";
input float theta;
bit[2] c;
qubit[2] q;
rx(theta) q[0];
rx(-theta) q[1];
c = measure q;
"""

init(target_profile=TargetProfile.Base)
import_openqasm(source, name="parameterized_program", program_type=ProgramType.File)


In [ ]:
from qdk.code.qasm_import import parameterized_program
from qdk.openqasm import compile

print(compile(parameterized_program, 1.57))

When running an OpenQASM program in simulation with qubit loss, additional `Result.Loss` values may be returned that indicate the measured qubit was lost during execution:

In [ ]:
init()

source = """
include "stdgates.inc";
bit[2] c;
qubit[2] q;
c = measure q;
"""

import_openqasm(source, name="measure2", program_type=ProgramType.File)

from qdk.code.qasm_import import measure2

Histogram(run(measure2, shots=1000, qubit_loss=0.1))

By using the special include `"qdk.inc"` you can check for loss at runtime using the `mresetz_checked` function. It returns an integer with two bits to indicate whether or not loss has occurred, such that `0` or `1` correspond to the qubit measurement and `2` corresponds to loss:

In [ ]:
source = """
include "stdgates.inc";
include "qdk.inc";
qubit q;
output int res;
h q;
res = mresetz_checked(q);
"""

import_openqasm(source, name="mresetz_checked_example", program_type=ProgramType.File)

from qdk.code.qasm_import import mresetz_checked_example

Histogram(run(mresetz_checked_example, shots=1000, qubit_loss=0.1))
